# 01. Linear Neuron and Invertible Coordinate Transforms

- **Category:** 00. Foundations & Classical Machine Learning
- **Lineage & Citations:** McCulloch & Pitts (1943), Rosenblatt (1958) *Perceptron*, Widrow & Hoff (1960) *Adaptive Linear Neuron (ADALINE)*.
- **Purpose:** Implements the fundamental univariate affine projection $f(x) = w x + b$ under Mean Squared Error (MSE), coupled with five invertible coordinate transformations (MinMax, Z-score, Parameterized Tanh, Parameterized Sigmoid, Parameterized DReLU).

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use an unadorned linear neuron when the true data-generating process exhibits intrinsic non-linear manifolds, multi-modal class densities, or when $D \gg N$ without rank regularization (where the normal equation $(X^\top X)^{-1}$ becomes ill-conditioned or singular).

## Mathematical Formulation & Complexity

### 1. Univariate Affine Map & OLS Normal Equations
Given observations $\mathbf{x} \in \mathbb{R}^N$ and targets $\mathbf{y} \in \mathbb{R}^N$, define the design matrix $\mathbf{X} = [\mathbf{x}, \mathbf{1}] \in \mathbb{R}^{N \times 2}$ and parameter vector $\boldsymbol{\theta} = [w, b]^\top \in \mathbb{R}^2$:
$$\hat{\mathbf{y}} = \mathbf{X} \boldsymbol{\theta}, \quad \mathcal{L}(\boldsymbol{\theta}) = \frac{1}{N} \|\mathbf{y} - \mathbf{X} \boldsymbol{\theta}\|_2^2$$

Setting $\nabla_{\boldsymbol{\theta}} \mathcal{L} = -\frac{2}{N} \mathbf{X}^\top (\mathbf{y} - \mathbf{X} \boldsymbol{\theta}) = \mathbf{0}$ yields the closed-form Ordinary Least Squares (OLS) estimator:
$$\boldsymbol{\theta}^* = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

The condition number $\kappa(\mathbf{X}^\top \mathbf{X}) = \frac{\sigma_{\max}(\mathbf{X}^\top \mathbf{X})}{\sigma_{\min}(\mathbf{X}^\top \mathbf{X})}$ dictates gradient descent convergence rate $\rho = \left( \frac{\kappa - 1}{\kappa + 1} \right)^2$.

### 2. Invertible Transforms and Exact Inverses
- **Affine Min-Max:** $t = 2 \frac{x - x_{\min}}{x_{\max} - x_{\min}} - 1 \iff x = \frac{t + 1}{2} (x_{\max} - x_{\min}) + x_{\min}$
- **Standard Z-Score:** $t = \frac{x - \mu}{\sigma} \iff x = t \cdot \sigma + \mu$
- **Parameterized Tanh:** $t = \tanh(s(x - c)) \iff x = \frac{\operatorname{arctanh}(\operatorname{clip}(t, -1+\epsilon, 1-\epsilon))}{s} + c$
- **Parameterized Sigmoid:** $t = \sigma(s(x - c)) \iff x = \frac{\operatorname{logit}(\operatorname{clip}(t, \epsilon, 1-\epsilon))}{s} + c$
- **Parameterized DReLU:** Linear within $[c+l, c+u]$, leaky with slope $\alpha > 0$ outside; inverted by $\alpha^{-1}$ scaling.

**Asymptotic Complexity:** Forward/Backward Time: $\mathcal{O}(N)$; OLS Inversion: $\mathcal{O}(N \cdot D^2 + D^3) = \mathcal{O}(N)$; Memory: $\mathcal{O}(N)$.

In [ ]:
import torch
import core

# 1. Deterministic Environment Setup
env = core.setup_env(seed=42)

# 2. Synthetic Micro-Data: y = 2x + 0 with Gaussian noise on [ -1, 5 ]
n_samples = 64
x_raw = torch.linspace(-1.0, 5.0, n_samples, dtype=torch.float32).unsqueeze(1)
true_w, true_b = 2.0, 0.0
noise = 0.05 * torch.randn_like(x_raw)
y_raw = true_w * x_raw + true_b + noise

print(f"Device: {env.device} | Dataset: X={x_raw.shape}, Y={y_raw.shape}")

In [ ]:
# 1. Invertible Coordinate Transforms Class
class InvertibleTransforms:
    @staticmethod
    def min_max(x: torch.Tensor) -> tuple[torch.Tensor, dict]:
        d_min, d_max = x.min(), x.max()
        diff = (d_max - d_min) if d_max != d_min else torch.tensor(1.0)
        t = 2.0 * (x - d_min) / diff - 1.0
        return t, {"min": d_min, "diff": diff}

    @staticmethod
    def undo_min_max(t: torch.Tensor, p: dict) -> torch.Tensor:
        return (t + 1.0) / 2.0 * p["diff"] + p["min"]

    @staticmethod
    def z_score(x: torch.Tensor) -> tuple[torch.Tensor, dict]:
        mu, std = x.mean(), x.std(unbiased=False).clamp_min(1e-7)
        return (x - mu) / std, {"mu": mu, "std": std}

    @staticmethod
    def undo_z_score(t: torch.Tensor, p: dict) -> torch.Tensor:
        return t * p["std"] + p["mu"]

    @staticmethod
    def tanh_transform(x: torch.Tensor, s: float = 1.0, c: float = 0.0) -> tuple[torch.Tensor, dict]:
        return torch.tanh(s * (x - c)), {"s": s, "c": c}

    @staticmethod
    def undo_tanh(t: torch.Tensor, p: dict, eps: float = 1e-6) -> torch.Tensor:
        clipped = t.clamp(-1.0 + eps, 1.0 - eps)
        return torch.arctanh(clipped) / p["s"] + p["c"]

# 2. First-Principles Linear Neuron
class RawLinearNeuron:
    def __init__(self, init_w: float = 0.5, init_b: float = 0.0):
        self.w = torch.tensor([[init_w]], dtype=torch.float32, requires_grad=True)
        self.b = torch.tensor([init_b], dtype=torch.float32, requires_grad=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.w + self.b

    def params(self) -> list[torch.Tensor]:
        return [self.w, self.b]

# 3. Closed-Form OLS Normal Equations Solver
def solve_ols(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    X_design = torch.cat([x, torch.ones_like(x)], dim=1)  # [N, 2]
    theta = torch.linalg.pinv(X_design.T @ X_design) @ X_design.T @ y
    return theta  # [w, b]

theta_ols = solve_ols(x_raw, y_raw)
print(f"OLS Normal Equation Solution: w={theta_ols[0].item():.4f}, b={theta_ols[1].item():.4f}")

In [ ]:
# Quantitative Verification & Invariant Checks
# 1. Invertibility Check: || x - undo(transform(x)) ||_inf < 1e-5
x_test = torch.linspace(-3.0, 3.0, 100, dtype=torch.float32).unsqueeze(1)

t_mm, p_mm = InvertibleTransforms.min_max(x_test)
rec_mm = InvertibleTransforms.undo_min_max(t_mm, p_mm)
torch.testing.assert_close(x_test, rec_mm, atol=1e-5, rtol=1e-5)

t_zs, p_zs = InvertibleTransforms.z_score(x_test)
rec_zs = InvertibleTransforms.undo_z_score(t_zs, p_zs)
torch.testing.assert_close(x_test, rec_zs, atol=1e-5, rtol=1e-5)

t_th, p_th = InvertibleTransforms.tanh_transform(x_test, s=0.5, c=0.0)
rec_th = InvertibleTransforms.undo_tanh(t_th, p_th)
torch.testing.assert_close(x_test, rec_th, atol=1e-4, rtol=1e-4)

# 2. Gradient Correctness Check via Autograd vs Analytic Formula
model = RawLinearNeuron(init_w=1.0, init_b=0.5)
pred = model.forward(x_raw)
loss = torch.mean((pred - y_raw) ** 2)
loss.backward()

# Analytical gradient: dL/dw = (2/N) * X^T (X w + b - y)
res = pred.detach() - y_raw
grad_w_analytic = (2.0 / len(x_raw)) * (x_raw.T @ res)
grad_b_analytic = (2.0 / len(x_raw)) * res.sum()
torch.testing.assert_close(model.w.grad, grad_w_analytic, atol=1e-5, rtol=1e-5)
torch.testing.assert_close(model.b.grad[0], grad_b_analytic, atol=1e-5, rtol=1e-5)

# 3. Equivalence: Gradient descent steps converge toward closed-form OLS
optimizer = torch.optim.SGD(model.params(), lr=0.05)
for _ in range(500):
    optimizer.zero_grad()
    l = torch.mean((model.forward(x_raw) - y_raw) ** 2)
    l.backward()
    optimizer.step()

torch.testing.assert_close(model.w.squeeze(), theta_ols[0].squeeze(), atol=1e-3, rtol=1e-3)
print("All mathematical invariant checks passed successfully.")

In [ ]:
# Standardized Performance Benchmark Table
from core.benchmark import benchmark_fn, format_benchmark_table

nn_linear = torch.nn.Linear(1, 1)
b_raw = benchmark_fn(lambda: model.forward(x_raw), iters=20, name="Raw Tensor Affine")
b_nn = benchmark_fn(lambda: nn_linear(x_raw), iters=20, name="torch.nn.Linear")
b_ols = benchmark_fn(lambda: solve_ols(x_raw, y_raw), iters=20, name="OLS Closed-Form")

print(format_benchmark_table([b_raw, b_nn, b_ols]))

In [ ]:
# Runnable Misuse Contrast: Unnormalized Inputs vs Normalized Invertible Flow
# FAILURE MODE: Input scale [1000, 5000] without normalization causes gradient
# explosion under high learning rates, or extreme saturation in bounded activations.
x_huge = x_raw * 1000.0
y_huge = 2.0 * x_huge

# 1. Unnormalized Training (Fails: Gradient norm explodes)
w_fail = torch.tensor([[0.5]], requires_grad=True)
b_fail = torch.tensor([0.0], requires_grad=True)
pred_fail = x_huge @ w_fail + b_fail
loss_fail = torch.mean((pred_fail - y_huge) ** 2)
loss_fail.backward()
grad_norm_fail = torch.sqrt(w_fail.grad**2 + b_fail.grad**2).item()

# 2. Normalized Training via Invertible Z-Score (Succeeds: Conditioned Gradients)
x_norm, p_x = InvertibleTransforms.z_score(x_huge)
y_norm, p_y = InvertibleTransforms.z_score(y_huge)
w_fixed = torch.tensor([[0.5]], requires_grad=True)
b_fixed = torch.tensor([0.0], requires_grad=True)
pred_fixed = x_norm @ w_fixed + b_fixed
loss_fixed = torch.mean((pred_fixed - y_norm) ** 2)
loss_fixed.backward()
grad_norm_fixed = torch.sqrt(w_fixed.grad**2 + b_fixed.grad**2).item()

print(f"[FAILURE] Unnormalized Gradient Norm: {grad_norm_fail:.2e} (Severe Explosion)")
print(f"[CORRECT] Z-Score Normalized Gradient Norm: {grad_norm_fixed:.4f} (Stable Conditioning)")
assert grad_norm_fail > 1e6 and grad_norm_fixed < 10.0

## Key Takeaways & Relative Navigation

### Key Takeaways
1. **Analytic Optimality:** The OLS normal equation $(X^\top X)^{-1} X^\top y$ provides the exact global minimum in one step without learning rates or iterations, but requires $X^\top X$ to be non-singular.
2. **Information Preservation:** Preprocessing transforms (Z-score, Min-Max, Tanh) are safe if and only if their Jacobian determinant $\det(J) \ne 0$ everywhere, enabling exact analytic recovery $x = f^{-1}(t)$.
3. **Conditioning:** Input normalization directly compresses the condition number $\kappa(X^\top X) \to 1$, eliminating ill-conditioned elliptical loss contours.

---
**Navigation:**  
&larr; [Master Index: README.md](../README.md) | [Index by Goal: INDEX_BY_GOAL.md](../INDEX_BY_GOAL.md) | [Next: 02. Tensors, Views, Strides & Memory Layouts](02_tensors_views_strides_and_memory_layouts.ipynb) &rarr;